In [16]:
import sympy as sp
from sympy.physics.hep.gamma_matrices import GammaMatrix as G, LorentzIndex, gamma_trace
from sympy.tensor.tensor import tensor_indices, tensor_heads

# ============================================================
# Utilidades
# ============================================================

def banner(title):
    print("\n" + "="*78)
    print(title)
    print("="*78)

def tensor_simplify(expr, g):
    """
    Simplificación tensorial robusta para expresiones de SymPy.
    Evita errores cuando la expresión ya es exactamente cero.
    """
    expr = sp.expand(expr)

    if expr == 0:
        return sp.Integer(0)

    if hasattr(expr, "contract_metric"):
        expr = expr.contract_metric(g)

    if expr == 0:
        return sp.Integer(0)

    if hasattr(expr, "canon_bp"):
        expr = expr.canon_bp()

    expr = sp.expand(expr)

    if expr == 0:
        return sp.Integer(0)

    return expr

# ============================================================
# Índices de Lorentz
# ============================================================

mu, nu = tensor_indices('mu nu', LorentzIndex)
a, b, c, d, r, s = tensor_indices('a b c d r s', LorentzIndex)

# Momentos como tensores de Lorentz
p1, p2, p3, p4 = tensor_heads('p1 p2 p3 p4', [LorentzIndex])

# Métrica
g = LorentzIndex.metric

# Masa del muón
m = sp.symbols('m', real=True)

# ============================================================
# Definición de slash
# ============================================================

ps1 = p1(a) * G(-a)
ps2 = p2(b) * G(-b)
ps3 = p3(c) * G(-c)
ps4 = p4(d) * G(-d)

banner("Definición de slash(p)")
print("slash(p1) =", ps1)
print("slash(p2) =", ps2)
print("slash(p3) =", ps3)
print("slash(p4) =", ps4)

# ============================================================
# 1) PRIMERA TRAZA DEL CANAL s
#    Tr( /p2 gamma^mu /p1 gamma^nu )
# ============================================================

banner("1) Primera traza: Tr( /p2 γ^μ /p1 γ^ν )")

T1 = gamma_trace(ps2 * G(mu) * ps1 * G(nu))
T1 = tensor_simplify(T1, g)

print("\nResultado de SymPy:")
print(T1)

dot21 = p2(r) * p1(-r)
T1_expected = 4 * (
    p2(mu) * p1(nu)
    + p2(nu) * p1(mu)
    - dot21 * g(mu, nu)
)
T1_expected = tensor_simplify(T1_expected, g)

print("\nResultado esperado:")
print(T1_expected)

print("\nDiferencia:")
print(tensor_simplify(T1 - T1_expected, g))

# ============================================================
# 2) SEGUNDA TRAZA DEL CANAL s
#    Tr[( /p3 + m ) gamma_mu ( /p4 - m ) gamma_nu]
# ============================================================

banner("2) Segunda traza: Tr[( /p3 + m ) γ_μ ( /p4 - m ) γ_ν]")

T2 = gamma_trace(ps3 * G(-mu) * ps4 * G(-nu)) \
     - m**2 * gamma_trace(G(-mu) * G(-nu))
T2 = tensor_simplify(T2, g)

print("\nResultado de SymPy:")
print(T2)

dot34 = p3(s) * p4(-s)
T2_expected = 4 * (
    p3(-mu) * p4(-nu)
    + p3(-nu) * p4(-mu)
    - (dot34 + m**2) * g(-mu, -nu)
)
T2_expected = tensor_simplify(T2_expected, g)

print("\nResultado esperado:")
print(T2_expected)

print("\nDiferencia:")
print(tensor_simplify(T2 - T2_expected, g))

# ============================================================
# 3) CONTRACCIÓN FINAL DEL CANAL s
# ============================================================

banner("3) Contracción final: T1^{μν} T2_{μν}")

contracted = tensor_simplify(T1 * T2, g)

print("\nResultado de SymPy:")
print(contracted)

dot13 = p1(a) * p3(-a)
dot24 = p2(b) * p4(-b)
dot14 = p1(c) * p4(-c)
dot23 = p2(d) * p3(-d)
dot12 = p1(r) * p2(-r)

contracted_expected = 32 * (
    dot13 * dot24
    + dot14 * dot23
    + m**2 * dot12
)
contracted_expected = tensor_simplify(contracted_expected, g)

print("\nResultado esperado:")
print(contracted_expected)

print("\nDiferencia:")
print(tensor_simplify(contracted - contracted_expected, g))



# ============================================================
# RESUMEN
# ============================================================

banner("Resumen teórico")

print("Primera traza:")
print("Tr(/p2 γ^μ /p1 γ^ν)")
print("= 4 [ p2^μ p1^ν + p2^ν p1^μ - (p1·p2) g^{μν} ]")

print("\nSegunda traza:")
print("Tr[(/p3 + m) γ_μ (/p4 - m) γ_ν]")
print("= 4 [ p3_μ p4_ν + p3_ν p4_μ - (p3·p4 + m^2) g_{μν} ]")

print("\nContracción:")
print("32 [ (p1·p3)(p2·p4) + (p1·p4)(p2·p3) + m^2 (p1·p2) ]")



banner("FIN")


Definición de slash(p)
slash(p1) = p1(L_0)*GammaMatrix(-L_0)
slash(p2) = p2(L_0)*GammaMatrix(-L_0)
slash(p3) = p3(L_0)*GammaMatrix(-L_0)
slash(p4) = p4(L_0)*GammaMatrix(-L_0)

1) Primera traza: Tr( /p2 γ^μ /p1 γ^ν )

Resultado de SymPy:
4*p1(mu)*p2(nu) + 4*p1(nu)*p2(mu) - 4*metric(mu, nu)*p1(L_0)*p2(-L_0)

Resultado esperado:
4*p1(mu)*p2(nu) + 4*p1(nu)*p2(mu) - 4*metric(mu, nu)*p1(L_0)*p2(-L_0)

Diferencia:
0

2) Segunda traza: Tr[( /p3 + m ) γ_μ ( /p4 - m ) γ_ν]

Resultado de SymPy:
-4*m**2*metric(-mu, -nu) + 4*p3(-mu)*p4(-nu) + 4*p3(-nu)*p4(-mu) - 4*metric(-mu, -nu)*p3(L_0)*p4(-L_0)

Resultado esperado:
-4*m**2*metric(-mu, -nu) + 4*p3(-mu)*p4(-nu) + 4*p3(-nu)*p4(-mu) - 4*metric(-mu, -nu)*p3(L_0)*p4(-L_0)

Diferencia:
0

3) Contracción final: T1^{μν} T2_{μν}

Resultado de SymPy:
32*m**2*p1(L_0)*p2(-L_0) + 32*p1(L_0)*p2(L_1)*p3(-L_0)*p4(-L_1) + 32*p1(L_0)*p2(L_1)*p3(-L_1)*p4(-L_0)

Resultado esperado:
32*m**2*p1(L_0)*p2(-L_0) + 32*p1(L_0)*p2(L_1)*p3(-L_0)*p4(-L_1) + 32*p1(L_0)*p2(L_1)

In [20]:
sp.latex(contracted_expected)

'32m^{2}p_{1}{}^{L_{0}}p_{2}{}_{L_{0}} + 32p_{1}{}^{L_{0}}p_{2}{}^{L_{1}}p_{3}{}_{L_{0}}p_{4}{}_{L_{1}} + 32p_{1}{}^{L_{0}}p_{2}{}^{L_{1}}p_{3}{}_{L_{1}}p_{4}{}_{L_{0}}'

In [21]:
from IPython.display import display, Math
display(Math(sp.latex(contracted_expected)))

<IPython.core.display.Math object>

In [22]:
from sympy.physics.hep.gamma_matrices import GammaMatrix as G, LorentzIndex, gamma_trace
from sympy.tensor.tensor import tensor_indices

mu, nu, rho, sigma = tensor_indices('mu nu rho, sigma', LorentzIndex)
mu, nu, rho, sigma = tensor_indices('mu nu rho, sigma', LorentzIndex)

#print(str(expr).replace("metric","g"))

gamma_trace(G(mu)*G(nu))

4*metric(mu, nu)

In [23]:
gamma_trace(G(mu)*G(nu)*G(rho)*G(sigma))

-4*metric(mu, rho)*metric(nu, sigma) + 4*metric(mu, nu)*metric(rho, sigma) + 4*metric(mu, sigma)*metric(nu, rho)

In [24]:
print(LorentzIndex.metric)

metric(LorentzIndex,LorentzIndex)


In [26]:
gamma5 = sp.Symbol('gamma5', commutative=False)
gamma_trace(gamma5*G(mu)*G(nu)*G(rho)*G(sigma))

gamma5*((-4)*metric(mu, rho)*metric(nu, sigma) + 4*metric(mu, nu)*metric(rho, sigma) + 4*metric(mu, sigma)*metric(nu, rho))